# Experiments 47-51 — Ayah-Level Clustering (Largest Scale Yet)

This is a significant scale-up from everything else in this project.
Every previous experiment used averaged, aggregated units (poets,
surahs, hizb, quarters). These five go down to the finest possible
level on the Quran side -- individual ayat, all 6,236 of them, no
averaging at all -- and compare that against poets at increasingly
fine granularity too.

- **47:** all 6,236 ayat, clustered alone (no poets)
- **48:** 6,236 ayat + all 260 poets, each poet kept separate (N=6,496)
- **49:** 6,236 ayat + the 35 well-documented poets, each separate (N=6,271)
- **50:** 6,236 ayat + the 19 poets, each separate (N=6,255)
- **51:** 6,236 ayat + 2,328 individual poems, each poem kept separate
  (not averaged up to poet level) (N=8,564)

**One assumption, stated clearly:** Experiment 51 said "2328 verses,"
but that exact number matches this project's established poem count
(2,328 poems across 260 poets), not individual verses -- verse-level
would be tens of thousands, not 2,328. Treating this as poem-level. If
that's wrong, let me know and I'll rebuild it as true verse-level.

**Given the scale, reporting is summary-level, not full enumeration**
-- with thousands of entities, printing every single cluster
membership wouldn't be a "short report" anymore. Full membership is
still saved to CSV for reference; what prints is cluster counts, sizes,
mixed-cluster proportions, and which poets/poems show up repeatedly.

**Runtime warning:** this is a genuinely bigger computational job than
anything else in this project. Reuses cached ayah/poet embeddings
where possible, but building fresh poem-level embeddings for
Experiment 51 (2,328 separate vectors, not the usual 260 poet-level
ones) is new and will take some time. Expect this notebook to run
noticeably longer than earlier ones -- possibly 15-30+ minutes total
depending on your machine, most of it in the clustering step itself
given the much larger N.

**Before you start:** put `poems.db` in the same folder as this
notebook — the real one, not an empty stand-in.

Run cells top to bottom, **Shift+Enter**. Be patient with cells that
take a while; a busy `[*]` next to a cell means it's still working, not
stuck.

In [ ]:
# CELL 1 -- Install packages
!pip -q install sentence-transformers torch scikit-learn umap-learn hdbscan pandas numpy matplotlib seaborn scipy requests

In [ ]:
# CELL 2 -- Configuration
import re, sqlite3, hashlib, json, warnings, pickle
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

warnings.filterwarnings("ignore")

DB_PATH = Path("poems.db")
CACHE_DIR = Path("quran_cache"); CACHE_DIR.mkdir(exist_ok=True)
EMBED_CACHE_DIR = Path("embed_cache"); EMBED_CACHE_DIR.mkdir(exist_ok=True)
FIGURES_DIR = Path("output/figures"); FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = Path("output/tables"); TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("output/reports"); REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if not DB_PATH.exists():
    raise FileNotFoundError(
        "poems.db not found in this folder. Put the real poems.db "
        "(same one used in every other notebook in this project) here."
    )
if DB_PATH.stat().st_size < 10_000:
    raise ValueError(
        f"poems.db is only {DB_PATH.stat().st_size} bytes -- this is almost "
        "certainly an empty stand-in file, not the real corpus database. "
        "Delete it and find the real poems.db (several hundred KB or more)."
    )

SBERT_MODEL_NAME = "akhooli/Arabic-SBERT-100K"
MAX_VERSES_PER_POEM = 20

UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_N_COMPONENTS_HIGH = 50
UMAP_METRIC = "cosine"
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
import torch
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("No GPU found -- will run on CPU. This notebook is larger-scale than "
          "earlier ones, so CPU runs may take noticeably longer -- be patient.")

def split_verses(poem_text):
    if not poem_text or not isinstance(poem_text, str):
        return []
    verses = re.split(r'[\n\r]+|[.!\u061F?\u061B;]+', poem_text)
    return [v.strip() for v in verses if len(v.strip()) > 10]

def normalize_word(w):
    w = re.sub(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED\u0670]", "", w)
    w = re.sub(r"\u0640", "", w)
    w = re.sub(r"[\u0622\u0623\u0625\u0671]", "\u0627", w)
    return w.strip()

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 11})
print(f"Config loaded. poems.db size: {DB_PATH.stat().st_size:,} bytes -- looks like a real file.")

In [ ]:
# CELL 3 -- Load the 260 poets and their individual poems from poems.db
conn = sqlite3.connect(str(DB_PATH))
df = pd.read_sql_query(
    "SELECT poet_name, poem_title, poem_text, poem_type, poem_meter, verses_count "
    "FROM poems WHERE poet_name IS NOT NULL AND poem_text IS NOT NULL "
    "AND LENGTH(poem_text) >= 50",
    conn,
)
conn.close()

df["poem_hash"] = df["poem_text"].apply(lambda t: hashlib.md5(t.strip().encode("utf-8")).hexdigest())
df = df.drop_duplicates(subset=["poem_hash"]).copy()
df = df.reset_index(drop=True)
df["poem_id"] = df.index  # stable unique ID per individual poem, for Experiment 51

poems_by_poet = defaultdict(list)
for _, row in df.iterrows():
    poems_by_poet[row["poet_name"]].append(row["poem_text"])
poems_by_poet = dict(poems_by_poet)

poet_total_verses = {
    poet: sum(len(split_verses(p)) for p in poems)
    for poet, poems in poems_by_poet.items()
}

print(f"Poets loaded: {len(poems_by_poet)}")
print(f"Individual poems loaded: {len(df)} (should be close to 2,328)")

In [ ]:
# CELL 4 -- Fetch Quran text (cached if available)
cache_file = CACHE_DIR / "quran_ayat_with_hizb.json"

if cache_file.exists():
    print("Loading Quran from local cache...")
    with open(cache_file, encoding="utf-8") as f:
        surahs_raw = json.load(f)
else:
    print("Fetching Quran from Al Quran Cloud API...")
    resp = requests.get("https://api.alquran.cloud/v1/quran/quran-uthmani", timeout=60)
    resp.raise_for_status()
    surahs_raw = resp.json()["data"]["surahs"]
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(surahs_raw, f, ensure_ascii=False)
    print("Fetched and cached.")

surah_names = {}
ayah_records = []
for s in surahs_raw:
    snum = s["number"]
    surah_names[snum] = s["englishName"]
    for a in s["ayahs"]:
        ayah_records.append({"surah_number": snum, "ayah_number": a["numberInSurah"], "text": a["text"]})

ayah_df = pd.DataFrame(ayah_records)
print(f"Surahs: {ayah_df['surah_number'].nunique()} | Ayat: {len(ayah_df)} (should be 6,236)")

In [ ]:
# CELL 5 -- Embed the 260 poets at POET level (cached, reused from earlier notebooks)
from sentence_transformers import SentenceTransformer

poet_cache_file = EMBED_CACHE_DIR / "poet_embeddings.pkl"

print("Loading model:", SBERT_MODEL_NAME)
model = SentenceTransformer(SBERT_MODEL_NAME)
print("Loaded.")

def _encode(texts, batch_size=64):
    if not texts:
        return np.array([])
    return model.encode(texts, batch_size=batch_size, show_progress_bar=False,
                         normalize_embeddings=True, convert_to_numpy=True)

def embed_poem_verse_average(poem_text, max_verses=MAX_VERSES_PER_POEM, seed=RANDOM_SEED, rng=None):
    rng = rng or np.random.RandomState(seed)
    verses = split_verses(poem_text)
    if len(verses) > max_verses:
        idx = rng.choice(len(verses), max_verses, replace=False)
        verses = [verses[i] for i in sorted(idx)]
    if not verses:
        return None
    return np.mean(_encode(verses), axis=0)

def embed_poet_verse_average(poems_by_author, max_verses=MAX_VERSES_PER_POEM, seed=RANDOM_SEED):
    out = {}
    rng = np.random.RandomState(seed)
    for author, poems in poems_by_author.items():
        poem_vectors = [embed_poem_verse_average(p, max_verses, rng=rng) for p in poems]
        poem_vectors = [v for v in poem_vectors if v is not None]
        if poem_vectors:
            out[author] = np.mean(poem_vectors, axis=0)
    return out

if poet_cache_file.exists():
    print("Loading cached poet embeddings...")
    with open(poet_cache_file, "rb") as f:
        poet_embeddings = pickle.load(f)
    print(f"Loaded {len(poet_embeddings)} cached poet embeddings.")
else:
    print("Embedding 260 poets at poet level (slow, one-time only)...")
    poet_embeddings = embed_poet_verse_average(poems_by_poet)
    with open(poet_cache_file, "wb") as f:
        pickle.dump(poet_embeddings, f)
    print(f"Done and cached. {len(poet_embeddings)} poets embedded.")

max_surah_verses = ayah_df.groupby("surah_number").size().max()
print(f"\nLongest Quran chapter: {max_surah_verses} ayat (computed fresh, not assumed)")
poets_35 = [p for p, v in poet_total_verses.items() if v > max_surah_verses]
poets_35_embeddings = {p: poet_embeddings[p] for p in poets_35 if p in poet_embeddings}
print(f"35-poet subset: {len(poets_35_embeddings)} poets")

In [ ]:
# CELL 6 -- Embed every ayah (cached), and build POEM-level embeddings (new --
# not the same as poet-level; each of the 2,328 poems gets kept as its own
# vector here, never averaged up to poet level)
ayah_embed_cache_file = EMBED_CACHE_DIR / "ayah_embeddings.pkl"

if ayah_embed_cache_file.exists():
    print("Loading cached ayah embeddings...")
    with open(ayah_embed_cache_file, "rb") as f:
        ayah_embeddings = pickle.load(f)
    print(f"Loaded {len(ayah_embeddings)} cached ayah embeddings.")
else:
    print(f"Embedding all {len(ayah_df)} ayat individually (slow, one-time only)...")
    all_texts = ayah_df["text"].tolist()
    all_vecs = _encode(all_texts, batch_size=64)
    ayah_embeddings = {}
    for (snum, anum), vec in zip(zip(ayah_df["surah_number"], ayah_df["ayah_number"]), all_vecs):
        ayah_embeddings[(snum, anum)] = vec
    with open(ayah_embed_cache_file, "wb") as f:
        pickle.dump(ayah_embeddings, f)
    print(f"Done and cached. {len(ayah_embeddings)} ayat embedded.")

ayat_units = {f"Ayah {s}:{a}": v for (s, a), v in ayah_embeddings.items()}
print(f"Ayah units ready for clustering: {len(ayat_units)}")

# --- Poem-level embeddings (new, for Experiment 51 only) ---
poem_embed_cache_file = EMBED_CACHE_DIR / "poem_embeddings.pkl"

if poem_embed_cache_file.exists():
    print("\nLoading cached poem-level embeddings...")
    with open(poem_embed_cache_file, "rb") as f:
        poem_embeddings = pickle.load(f)
    print(f"Loaded {len(poem_embeddings)} cached poem embeddings.")
else:
    print(f"\nEmbedding all {len(df)} individual poems at poem level "
          f"(new, slow, one-time only -- this is the big new step for this notebook)...")
    rng = np.random.RandomState(RANDOM_SEED)
    poem_embeddings = {}
    for i, row in df.iterrows():
        vec = embed_poem_verse_average(row["poem_text"], rng=rng)
        if vec is not None:
            poem_embeddings[int(row["poem_id"])] = vec
        if (i + 1) % 300 == 0:
            print(f"  {i+1}/{len(df)} poems embedded...")
    with open(poem_embed_cache_file, "wb") as f:
        pickle.dump(poem_embeddings, f)
    print(f"Done and cached. {len(poem_embeddings)} poems embedded.")

print(f"Poem-level embeddings ready: {len(poem_embeddings)} (should be close to 2,328)")

In [ ]:
# CELL 7 -- Clustering helper, then identify the 19-poet subset
import umap, hdbscan
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score

def run_umap(matrix, n_components, n_neighbors, min_dist, metric="cosine", random_state=RANDOM_SEED):
    reducer = umap.UMAP(n_neighbors=min(n_neighbors, len(matrix) - 1), n_components=n_components,
                         min_dist=min_dist, metric=metric, random_state=random_state)
    return reducer.fit_transform(matrix)

def cluster_and_report(embeddings_dict, label_types, title, n_neighbors=UMAP_N_NEIGHBORS,
                       min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, verbose=True):
    names = sorted(embeddings_dict.keys(), key=str)
    n = len(names)
    print(f"[{title}] Building matrix for {n} entities...")
    emb_matrix = np.array([embeddings_dict[nm] for nm in names])
    print(f"[{title}] Running UMAP (this is the slow step at this scale)...")
    umap_high = run_umap(emb_matrix, min(UMAP_N_COMPONENTS_HIGH, max(2, n - 2)), n_neighbors, 0.0, UMAP_METRIC)
    print(f"[{title}] Running HDBSCAN...")
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min(min_cluster_size, max(2, n // 10)),
                                min_samples=HDBSCAN_MIN_SAMPLES,
                                cluster_selection_method="eom", metric="euclidean")
    labels = clusterer.fit_predict(umap_high)
    umap_2d = run_umap(emb_matrix, 2, n_neighbors, UMAP_MIN_DIST, UMAP_METRIC)
    mask = labels >= 0
    n_clusters = len(set(labels[mask])) if mask.sum() > 0 else 0
    n_outliers = int(np.sum(labels == -1))
    sil = float(silhouette_score(umap_high[mask], labels[mask])) if n_clusters >= 2 and mask.sum() > n_clusters else None
    result_df = pd.DataFrame({"name": names, "type": [label_types[nm] for nm in names],
        "cluster": labels, "umap_x": umap_2d[:, 0], "umap_y": umap_2d[:, 1]})
    if verbose:
        print(f"=== {title} ===")
        print(f"Total: {n} | Clusters: {n_clusters} | Outliers: {n_outliers} | Silhouette: {sil}")
    return result_df, {"n_entities": n, "n_clusters": n_clusters, "n_outliers": n_outliers, "silhouette": sil}

print("Identifying the 19-poet subset by rerunning Experiment 1 (all 260 poets + Quran whole)...")
surah_vectors_tmp = {}
for snum in surah_names:
    keys = list(zip(ayah_df[ayah_df["surah_number"] == snum]["surah_number"],
                    ayah_df[ayah_df["surah_number"] == snum]["ayah_number"]))
    vecs = [ayah_embeddings[k] for k in keys if k in ayah_embeddings]
    if vecs:
        surah_vectors_tmp[snum] = np.mean(vecs, axis=0)
quran_whole_vector = np.mean(list(surah_vectors_tmp.values()), axis=0)

exp1_rerun_embeddings = dict(poet_embeddings)
exp1_rerun_embeddings["Quran (whole)"] = quran_whole_vector
names_1 = sorted(exp1_rerun_embeddings.keys())
n_1 = len(names_1)
emb_matrix_1 = np.array([exp1_rerun_embeddings[nm] for nm in names_1])
umap_high_1 = run_umap(emb_matrix_1, min(UMAP_N_COMPONENTS_HIGH, n_1 - 2), UMAP_N_NEIGHBORS, 0.0)
clusterer_1 = hdbscan.HDBSCAN(min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, min_samples=HDBSCAN_MIN_SAMPLES,
                              cluster_selection_method="eom", metric="euclidean")
labels_1 = clusterer_1.fit_predict(umap_high_1)
labels_1_by_name = dict(zip(names_1, labels_1))
quran_cluster_1 = labels_1_by_name["Quran (whole)"]
poets_19_names = [nm for nm in names_1 if nm != "Quran (whole)" and labels_1_by_name[nm] == quran_cluster_1]
poets_19_embeddings = {p: poet_embeddings[p] for p in poets_19_names}
print(f"19-poet subset identified: {len(poets_19_embeddings)} poets "
      "(small variation from the original 19 is normal run-to-run noise).")

## Experiment 47 — All 6,236 Ayat, Clustered Alone

In [ ]:
# CELL 8 -- Experiment 47: all 6,236 ayat, clustered alone
exp47_types = {nm: "Ayah" for nm in ayat_units}
exp47_df, exp47_metrics = cluster_and_report(ayat_units, exp47_types, "Experiment 47: 6,236 ayat alone")
exp47_df.to_csv(TABLES_DIR / "experiment47_full_membership.csv", index=False)

cluster_sizes_47 = exp47_df[exp47_df["cluster"] != -1]["cluster"].value_counts().sort_index()
print(f"\nCluster size summary (top 10 largest):")
print(cluster_sizes_47.sort_values(ascending=False).head(10).to_string())
print(f"\nSmallest cluster size: {cluster_sizes_47.min() if len(cluster_sizes_47) else 'n/a'}")
print(f"Largest cluster size: {cluster_sizes_47.max() if len(cluster_sizes_47) else 'n/a'}")

## Experiments 48-50 — All 6,236 Ayat vs. Individual Poets (260, then 35, then 19)

In [ ]:
# CELL 9 -- Experiments 48, 49, 50: ayat vs individual poets, three subsets
def cluster_ayat_vs_poets(poet_dict, poet_count_label, exp_num, title_note):
    embeddings = {**poet_dict, **ayat_units}
    types = {nm: ("Poet" if nm in poet_dict else "Ayah") for nm in embeddings}
    result_df, metrics = cluster_and_report(embeddings, types,
        f"Experiment {exp_num}: {poet_count_label} poets + 6,236 ayat ({title_note})")

    comp = result_df.groupby("cluster")["type"].value_counts().unstack(fill_value=0)
    comp_real = comp[comp.index != -1]  # exclude noise bucket from "mixed" (same fix as Exp 41-44)
    mixed = comp_real[(comp_real.get("Poet", 0) > 0) & (comp_real.get("Ayah", 0) > 0)]
    ayah_rows = result_df[result_df["type"] == "Ayah"]
    ayat_in_mixed = int(sum(comp_real.loc[mixed.index, "Ayah"])) if len(mixed) > 0 else 0
    ayah_outliers = len(ayah_rows[ayah_rows["cluster"] == -1])

    print(f"\nAyat in clusters shared with >=1 poet: {ayat_in_mixed} / {len(ayah_rows)} "
          f"({ayat_in_mixed/len(ayah_rows)*100:.1f}%)")
    print(f"Ayat classified as noise: {ayah_outliers} / {len(ayah_rows)}")
    print(f"Mixed clusters: {len(mixed)}")

    if len(mixed) > 0:
        # Which poets appear across mixed clusters, ranked by how many
        # mixed clusters they show up in (a poet in many mixed clusters
        # is a recurring pattern worth naming; listing every individual
        # ayah in each cluster would not be, at this scale)
        poet_appearance_counts = Counter()
        for cl in mixed.index:
            members = result_df[result_df["cluster"] == cl]
            for poet_name in members[members["type"] == "Poet"]["name"]:
                poet_appearance_counts[poet_name] += 1
        print("Poets appearing in mixed clusters (poet: number of mixed clusters they're in):")
        for poet_name, count in poet_appearance_counts.most_common(15):
            print(f"  {poet_name}: {count}")

    result_df.to_csv(TABLES_DIR / f"experiment{exp_num}_full_membership.csv", index=False)
    return result_df, metrics, {"ayat_in_mixed": ayat_in_mixed, "n_ayat": len(ayah_rows),
                                "n_ayah_outliers": ayah_outliers, "n_mixed_clusters": len(mixed)}

exp48_df, exp48_metrics, exp48_summary = cluster_ayat_vs_poets(poet_embeddings, 260, 48, "N=6,496")
print("\n" + "=" * 70)
exp49_df, exp49_metrics, exp49_summary = cluster_ayat_vs_poets(poets_35_embeddings, 35, 49, "N=6,271")
print("\n" + "=" * 70)
exp50_df, exp50_metrics, exp50_summary = cluster_ayat_vs_poets(poets_19_embeddings, 19, 50, "N=6,255")

## Experiment 51 — All 6,236 Ayat vs. 2,328 Individual Poems

Each poem kept as its own entity, not averaged up to poet level --
the largest single clustering job in this project (N=8,564).

In [ ]:
# CELL 10 -- Experiment 51: 6,236 ayat vs 2,328 individual poems (N=8,564)
poem_id_to_poet = dict(zip(df["poem_id"], df["poet_name"]))
poem_units = {f"Poem {pid}": v for pid, v in poem_embeddings.items()}

embeddings_51 = {**poem_units, **ayat_units}
types_51 = {nm: ("Poem" if nm in poem_units else "Ayah") for nm in embeddings_51}
exp51_df, exp51_metrics = cluster_and_report(embeddings_51, types_51,
    "Experiment 51: 2,328 poems + 6,236 ayat (N=8,564)")

comp51 = exp51_df.groupby("cluster")["type"].value_counts().unstack(fill_value=0)
comp51_real = comp51[comp51.index != -1]
mixed51 = comp51_real[(comp51_real.get("Poem", 0) > 0) & (comp51_real.get("Ayah", 0) > 0)]
ayah_rows_51 = exp51_df[exp51_df["type"] == "Ayah"]
ayat_in_mixed_51 = int(sum(comp51_real.loc[mixed51.index, "Ayah"])) if len(mixed51) > 0 else 0
ayah_outliers_51 = len(ayah_rows_51[ayah_rows_51["cluster"] == -1])

print(f"\nAyat in clusters shared with >=1 poem: {ayat_in_mixed_51} / {len(ayah_rows_51)} "
      f"({ayat_in_mixed_51/len(ayah_rows_51)*100:.1f}%)")
print(f"Ayat classified as noise: {ayah_outliers_51} / {len(ayah_rows_51)}")
print(f"Mixed clusters: {len(mixed51)}")

if len(mixed51) > 0:
    poet_appearance_counts_51 = Counter()
    for cl in mixed51.index:
        members = exp51_df[exp51_df["cluster"] == cl]
        for poem_name in members[members["type"] == "Poem"]["name"]:
            pid = int(poem_name.replace("Poem ", ""))
            poet_appearance_counts_51[poem_id_to_poet.get(pid, "unknown")] += 1
    print("\nPoets whose poems appear in mixed clusters (poet: number of poems involved):")
    for poet_name, count in poet_appearance_counts_51.most_common(15):
        print(f"  {poet_name}: {count}")

exp51_df.to_csv(TABLES_DIR / "experiment51_full_membership.csv", index=False)
exp51_summary = {"ayat_in_mixed": ayat_in_mixed_51, "n_ayat": len(ayah_rows_51),
                 "n_ayah_outliers": ayah_outliers_51, "n_mixed_clusters": len(mixed51)}

In [ ]:
# CELL 11 -- Figures (using small points given the very large N in these plots)
def plot_large(df, title, save_name, color_by_type=None):
    fig, ax = plt.subplots(figsize=(12, 9))
    unique_clusters = sorted(df["cluster"].unique())
    n_clust_plot = len([c for c in unique_clusters if c >= 0])
    colors = plt.cm.tab20(np.linspace(0, 1, max(n_clust_plot, 1)))
    for cl in unique_clusters:
        sub = df[df["cluster"] == cl]
        color = "gray" if cl == -1 else colors[cl % len(colors)]
        if color_by_type:
            for t, marker, size in color_by_type:
                tsub = sub[sub["type"] == t]
                if len(tsub) > 0:
                    ax.scatter(tsub["umap_x"], tsub["umap_y"], c=[color], marker=marker, s=size,
                              alpha=0.5, edgecolors="none")
        else:
            ax.scatter(sub["umap_x"], sub["umap_y"], c=[color], marker="o", s=8, alpha=0.5, edgecolors="none")
    ax.set_xlabel("UMAP Dimension 1"); ax.set_ylabel("UMAP Dimension 2")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / save_name, dpi=300, bbox_inches="tight")
    plt.show()

plot_large(exp47_df, "Experiment 47: 6,236 Ayat Alone", "experiment47_umap.png")
plot_large(exp48_df, "Experiment 48: 260 Poets + 6,236 Ayat", "experiment48_umap.png",
          color_by_type=[("Ayah", ".", 6), ("Poet", "^", 40)])
plot_large(exp49_df, "Experiment 49: 35 Poets + 6,236 Ayat", "experiment49_umap.png",
          color_by_type=[("Ayah", ".", 6), ("Poet", "^", 60)])
plot_large(exp50_df, "Experiment 50: 19 Poets + 6,236 Ayat", "experiment50_umap.png",
          color_by_type=[("Ayah", ".", 6), ("Poet", "^", 60)])
plot_large(exp51_df, "Experiment 51: 2,328 Poems + 6,236 Ayat", "experiment51_umap.png",
          color_by_type=[("Ayah", ".", 5), ("Poem", "^", 12)])

print("All figures saved.")

In [ ]:
# CELL 12 -- Final report
report_path = REPORTS_DIR / "experiments_47_51_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write("EXPERIMENTS 47-51: AYAH-LEVEL CLUSTERING (LARGEST SCALE)\n")
    f.write("=" * 70 + "\n\n")

    f.write("EXPERIMENT 47: 6,236 AYAT ALONE\n" + "-" * 40 + "\n")
    for k, v in exp47_metrics.items():
        f.write(f"  {k}: {v}\n")
    f.write(f"  Largest cluster: {cluster_sizes_47.max() if len(cluster_sizes_47) else 'n/a'}\n")
    f.write(f"  Smallest cluster: {cluster_sizes_47.min() if len(cluster_sizes_47) else 'n/a'}\n\n")

    for exp_num, metrics, summary, label in [
        (48, exp48_metrics, exp48_summary, "260 poets + 6,236 ayat"),
        (49, exp49_metrics, exp49_summary, "35 poets + 6,236 ayat"),
        (50, exp50_metrics, exp50_summary, "19 poets + 6,236 ayat"),
        (51, exp51_metrics, exp51_summary, "2,328 poems + 6,236 ayat"),
    ]:
        f.write(f"EXPERIMENT {exp_num}: {label}\n" + "-" * 40 + "\n")
        for k, v in metrics.items():
            f.write(f"  {k}: {v}\n")
        f.write(f"  Ayat in mixed clusters: {summary['ayat_in_mixed']} / {summary['n_ayat']} "
                f"({summary['ayat_in_mixed']/summary['n_ayat']*100:.1f}%)\n")
        f.write(f"  Ayat classified as noise: {summary['n_ayah_outliers']} / {summary['n_ayat']}\n")
        f.write(f"  Mixed clusters: {summary['n_mixed_clusters']}\n\n")

print(f"Report written to {report_path}")
print()
print(open(report_path, encoding="utf-8").read())

## Done

Output in `output/`:
- `output/tables/experiment47/48/49/50/51_full_membership.csv` — full
  per-entity cluster assignment for every experiment
- `output/figures/` — UMAP plots for all five
- `output/reports/experiments_47_51_report.txt` — everything together

Send this back and I'll write the report.